In [19]:
import pandas as pd
import numpy as np
import ast
import re
import nltk
from nltk.corpus import stopwords
import gensim.downloader as api

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Descargar stopwords de NLTK
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\marco\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# 1. Cargar el dataset
df = pd.read_csv('/workspace/tmdb_5000_movies.csv')

# Función para extraer el primer género de la cadena JSON
def extract_main_genre(genre_str):
    try:
        genres = ast.literal_eval(genre_str)
        if len(genres) > 0:
            return genres[0]['name']
        return None
    except:
        return None

# Aplicar la función y filtrar columnas relevantes
df['main_genre'] = df['genres'].apply(extract_main_genre)
df = df[['overview', 'main_genre']]

# 2. Eliminar filas con valores nulos
df = df.dropna(subset=['overview', 'main_genre']).reset_index(drop=True)

print(f"Total de películas después de limpieza: {len(df)}")
print(df.head(3))

Total de películas después de limpieza: 4772
                                            overview main_genre
0  In the 22nd century, a paraplegic Marine is di...     Action
1  Captain Barbossa, long believed to be dead, ha...  Adventure
2  A cryptic message from Bond’s past sends him o...     Action


In [21]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Convertir a minúsculas
    text = text.lower()
    # Eliminar caracteres especiales y puntuación (mantener solo letras y espacios)
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenización simple y eliminación de stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    # Reconstruir el string (requerido para el Tokenizer de Keras más adelante)
    return ' '.join(tokens)

df['clean_overview'] = df['overview'].apply(preprocess_text)
print("Ejemplo de texto limpio:")
print(df['clean_overview'].iloc[0])

Ejemplo de texto limpio:
nd century paraplegic marine dispatched moon pandora unique mission becomes torn following orders protecting alien civilization


In [22]:
print("Cargando modelo GloVe...")
glove_model = api.load("glove-wiki-gigaword-100")
print("¡Modelo GloVe cargado exitosamente!")

Cargando modelo GloVe...
¡Modelo GloVe cargado exitosamente!


In [23]:
# Codificación de variables objetivo (Géneros)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['main_genre'])
num_classes = len(label_encoder.classes_)

# Configuración del Tokenizer
max_words = 10000  # Tamaño del vocabulario
max_len = 100      # Longitud máxima de las secuencias (sinopsis)

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['clean_overview'])

# Convertir textos a secuencias de enteros y aplicar Padding
X_seq = tokenizer.texts_to_sequences(df['clean_overview'])
X_pad = pad_sequences(X_seq, maxlen=max_len, padding='post')

# Dividir en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_pad, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Clases únicas: {num_classes}")

Dimensiones de X_train: (3817, 100)
Clases únicas: 20


In [24]:
# Crear matriz de embeddings inicializada con ceros
embedding_dim = 100
word_index = tokenizer.word_index
embedding_matrix = np.zeros((max_words, embedding_dim))

# Llenar la matriz con los vectores de GloVe
for word, i in word_index.items():
    if i < max_words:
        try:
            embedding_matrix[i] = glove_model[word]
        except KeyError:
            pass # Si la palabra no está en GloVe, se queda como vector de ceros

# Definir la arquitectura del modelo
model = Sequential([
    # Capa Embedding con los pesos de GloVe congelados (trainable=False)
    Embedding(input_dim=max_words, 
              output_dim=embedding_dim, 
              weights=[embedding_matrix], 
              input_length=max_len, 
              trainable=False),
    
    LSTM(128, return_sequences=False),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# Compilación con Optimizador Adam
optimizer = Adam(learning_rate=0.001)
model.compile(loss='sparse_categorical_crossentropy', 
              optimizer=optimizer, 
              metrics=['accuracy'])

model.summary()

c:\Users\marco\.conda\envs\unir\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,000,000 (3.81 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 1,000,000 (3.81 MB)

In [25]:
# Calcular pesos de clase balanceados
class_weights = compute_class_weight(class_weight='balanced', 
                                     classes=np.unique(y_train), 
                                     y=y_train)
class_weight_dict = dict(enumerate(class_weights))

# Entrenar el modelo con los hiperparámetros solicitados
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test, y_test),
    class_weight=class_weight_dict
)

Epoch 1/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - accuracy: 0.0147 - loss: 2.9977 - val_accuracy: 0.1560 - val_loss: 2.9905
Epoch 2/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 55ms/step - accuracy: 0.0621 - loss: 3.0046 - val_accuracy: 0.2513 - val_loss: 2.9839
Epoch 3/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.0990 - loss: 3.0014 - val_accuracy: 0.2534 - val_loss: 2.9622
Epoch 4/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.0443 - loss: 2.9854 - val_accuracy: 0.0188 - val_loss: 2.9918
Epoch 5/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.0275 - loss: 2.9965 - val_accuracy: 0.0188 - val_loss: 2.9897
Epoch 6/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.0296 - loss: 2.9966 - val_accuracy: 0.0702 - val_loss: 2.9884
Epoch 7/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 55ms/step - accuracy: 0.0461 - loss: 2.9466 - val_accuracy: 0.0105 - val_loss: 2.9932
Epoch 8/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - accuracy: 0.0532 - loss: 2.9204 - val_accuracy: 0.0042 - v

In [26]:
# Predicciones
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Reporte de Clasificación
target_names = label_encoder.classes_
# AÑADIDO: Definimos explícitamente los índices de las clases (del 0 al 19)
labels_indices = np.arange(len(target_names))

print("\n--- REPORTE DE CLASIFICACIÓN ---")
# AÑADIDO: Pasamos el parámetro labels=labels_indices
print(classification_report(y_test, y_pred, 
                            labels=labels_indices, 
                            target_names=target_names, 
                            zero_division=0))

# Reflexión de mejora (Para documentar en tu notebook):
print("\nNota para análisis: Las clases minoritarias suelen tener un mejor 'Recall' gracias al uso de 'class_weight'.")
print("El 'Dropout' al 50% ayuda a que la pérdida en validación no se dispare tan rápido respecto al entrenamiento.")

30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step

--- REPORTE DE CLASIFICACIÓN ---
                 precision    recall  f1-score   support

         Action       0.00      0.00      0.00       151
      Adventure       0.00      0.00      0.00        68
      Animation       0.00      0.00      0.00        25
         Comedy       0.00      0.00      0.00       209
          Crime       0.00      0.00      0.00        39
    Documentary       0.08      0.12      0.10        17
          Drama       0.25      0.00      0.01       241
         Family       0.00      0.00      0.00        11
        Fantasy       0.00      0.00      0.00        24
        Foreign       0.00      0.00      0.00         0
        History       0.00      0.00      0.00         5
         Horror       0.06      0.98      0.12        60
          Music       0.00      0.00      0.00         7
        Mystery       0.00      0.00      0.00         8
        Romance       0.00      0.00      0.00        21
Science Ficti